# CVBoost - Sprint 3 : Fine-Tuning LoRA du modele NLLB

Ce notebook entraine le modele NLLB a traduire du francais vers le lingala.


In [ ]:
# Verification que le GPU est disponible
import torch

if torch.cuda.is_available():
    print('GPU disponible :', torch.cuda.get_device_name(0))
else:
    print('Pas de GPU. Aller dans Execution > Modifier le type execution > T4 GPU')

GPU disponible : Tesla T4


In [ ]:
# Connexion a Google Drive pour sauvegarder le modele
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/CvBoost3/checkpoints/nllb-lora'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Dossier de sauvegarde :', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dossier de sauvegarde : /content/drive/MyDrive/CvBoost3/checkpoints/nllb-lora


In [ ]:
# Installation des dépendances du fine-tuning LoRA
%pip install -q \
    transformers==4.45.0 \
    peft==0.12.0 \
    datasets==2.20.0 \
    sacrebleu==2.4.0 \
    accelerate==0.34.0 \
    sentencepiece \
    protobuf

print("Installation terminée.")

Installation terminée.


In [ ]:
import torch
import transformers
import peft
import datasets
import huggingface_hub
import fsspec

print("PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
print("Transformers :", transformers.__version__)
print("PEFT :", peft.__version__)
print("Datasets :", datasets.__version__)
print("Hugging Face Hub :", huggingface_hub.__version__)
print("Fsspec :", fsspec.__version__)

PyTorch : 2.11.0+cu130
GPU disponible : True
Transformers : 4.45.0
PEFT : 0.17.1
Datasets : 2.20.0
Hugging Face Hub : 0.36.2
Fsspec : 2024.5.0


In [ ]:
# 1. Vérifier l'environnement GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU indisponible. Dans Colab : Exécution > Modifier le type d'exécution > T4 GPU, puis relancer."
    )

print("GPU :", torch.cuda.get_device_name(0))
print("PyTorch :", torch.__version__)


GPU : Tesla T4
PyTorch : 2.11.0+cu130


In [ ]:
# 2. Monter Google Drive et définir le dossier de sortie
from google.colab import drive
drive.mount("/content/drive")

import os
OUTPUT_DIR = "/content/drive/MyDrive/CVBoost/checkpoints/nllb-lora-lin-fra"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Sortie :", OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sortie : /content/drive/MyDrive/CVBoost/checkpoints/nllb-lora-lin-fra


In [ ]:
# 3. Installer des versions cohérentes pour ce notebook
%pip install -q transformers==4.45.0 peft==0.12.0 datasets==2.20.0 sacrebleu==2.4.0 accelerate==0.34.0 sentencepiece protobuf

print("Installation terminée. Si Colab demande de redémarrer le runtime, faites-le puis reprenez à la cellule 1.")


Installation terminée. Si Colab demande de redémarrer le runtime, faites-le puis reprenez à la cellule 1.


In [ ]:
from datasets import load_dataset

DATASET_NAME = "CLEAR-Global/Gamayun-kits"
DATASET_CONFIG = "lin-fra"

raw = load_dataset(DATASET_NAME, DATASET_CONFIG)

print(raw)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    kit5k: Dataset({
        features: ['fra', 'lin'],
        num_rows: 5000
    })
})


In [ ]:
# 1. Sélectionner les 5 000 paires français-lingala
ds = raw["kit5k"]

# 2. Renommer les colonnes pour faciliter le travail
ds = ds.rename_column("fra", "source")
ds = ds.rename_column("lin", "target")

# 3. Supprimer les lignes vides ou incomplètes
ds = ds.filter(
    lambda x: x["source"] is not None
    and x["target"] is not None
    and len(x["source"].strip()) > 0
    and len(x["target"].strip()) > 0
)

# 4. Nettoyer les espaces inutiles
ds = ds.map(lambda x: {
    "source": x["source"].strip(),
    "target": x["target"].strip()
})

# 5. Vérifier le résultat
print("Nombre de paires utilisables :", len(ds))
print("Colonnes :", ds.column_names)
print("Exemple de paire :", ds[0])

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Nombre de paires utilisables : 5000
Colonnes : ['source', 'target']
Exemple de paire : {'source': 'Où ces photos ont-elles été prises\xa0?', 'target': 'Bakangaki bilili oyo wapi ?'}


In [ ]:
from datasets import DatasetDict
import random

# Fixer la graine pour obtenir une séparation reproductible
random.seed(42)

# Mélanger les indices des 5 000 exemples
indices = list(range(len(ds)))
random.shuffle(indices)

# Calculer les tailles des trois groupes
n = len(indices)
n_train = int(n * 0.80)
n_val = int(n * 0.10)

# Construire les trois groupes
splits = DatasetDict({
    "train": ds.select(indices[:n_train]),
    "validation": ds.select(indices[n_train:n_train + n_val]),
    "test": ds.select(indices[n_train + n_val:])
})

# Afficher les résultats
print("Entraînement :", len(splits["train"]))
print("Validation :", len(splits["validation"]))
print("Test :", len(splits["test"]))

Entraînement : 4000
Validation : 500
Test : 500


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"

# Charger le tokenizer associé à NLLB
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="fra_Latn",
    tgt_lang="lin_Latn"
)

print("Tokenizer chargé avec succès !")
print("Langue source :", tokenizer.src_lang)
print("Langue cible :", tokenizer.tgt_lang)

Tokenizer chargé avec succès !
Langue source : fra_Latn
Langue cible : lin_Latn


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM

# Vérifier si le GPU est disponible
print("GPU disponible :", torch.cuda.is_available())

# Charger le modèle NLLB
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("Modèle NLLB chargé avec succès !")
print("Type du modèle :", model.config.model_type)
print("Nombre de paramètres :", sum(p.numel() for p in model.parameters()))

GPU disponible : True
Modèle NLLB chargé avec succès !
Type du modèle : m2m_100
Nombre de paramètres : 615073792


In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"

print("Modèle sélectionné :", MODEL_NAME)

Modèle sélectionné : facebook/nllb-200-distilled-600M


In [ ]:
from transformers import AutoModelForSeq2SeqLM
import torch

print("Chargement du modèle NLLB...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Modèle chargé sur :", device)

Chargement du modèle NLLB...
Modèle chargé sur : cuda


In [ ]:
import torch

print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU utilisé :", torch.cuda.get_device_name(0))
else:
    print("Aucun GPU détecté")

GPU disponible : True
GPU utilisé : Tesla T4


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Configuration de LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Appliquer LoRA au modèle NLLB
model = get_peft_model(model, lora_config)

# Afficher les paramètres entraînables
model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 616,253,440 || trainable%: 0.1914


In [ ]:
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

def tokenize_batch(batch):
    # Transformer les phrases françaises en tokens
    model_inputs = tokenizer(
        batch["source"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    )

    # Transformer les traductions lingala en tokens cibles
    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


# Appliquer la tokenisation aux trois groupes
print("Tokenisation en cours...")

tokenized = splits.map(
    tokenize_batch,
    batched=True,
    remove_columns=splits["train"].column_names
)

print("Tokenisation terminée !")
print(tokenized)

Tokenisation en cours...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenisation terminée !
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
})


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

print("DataCollator prêt !")

DataCollator prêt !


In [ ]:
from transformers import Seq2SeqTrainingArguments

# Forcer le modèle à générer en lingala
target_language_id = tokenizer.convert_tokens_to_ids("lin_Latn")

model.config.forced_bos_token_id = target_language_id
model.generation_config.forced_bos_token_id = target_language_id

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-lora-fr-lin",

    # Durée de l'entraînement
    num_train_epochs=3,

    # Taille des lots
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    # Vitesse d'apprentissage
    learning_rate=2e-4,
    weight_decay=0.01,

    # Évaluation et sauvegarde
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # Suivi de l'entraînement
    logging_strategy="steps",
    logging_steps=50,

    # Optimisation GPU
    fp16=torch.cuda.is_available(),

    # Évaluation du meilleur modèle
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none"
)

print("Configuration de l'entraînement terminée !")
print("Langue cible : lingala")
print("Nombre d'époques :", training_args.num_train_epochs)
print("GPU disponible :", torch.cuda.is_available())

Configuration de l'entraînement terminée !
Langue cible : lingala
Nombre d'époques : 3
GPU disponible : True


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
from transformers import Seq2SeqTrainer

# Création du Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("Trainer créé avec succès !")
print("Exemples d'entraînement :", len(trainer.train_dataset))
print("Exemples de validation :", len(trainer.eval_dataset))

Trainer créé avec succès !
Exemples d'entraînement : 4000
Exemples de validation : 500


/usr/local/lib/python3.13/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
print("Début de l'entraînement du modèle NLLB français-lingala...")

train_result = trainer.train()

print("Entraînement terminé !")
print(train_result)

Début de l'entraînement du modèle NLLB français-lingala...


Epoch,Training Loss,Validation Loss
1,2.090000,2.064329
2,2.030000,2.039728
3,2.034500,2.031438


Entraînement terminé !
TrainOutput(global_step=750, training_loss=2.113566182454427, metrics={'train_runtime': 385.8819, 'train_samples_per_second': 31.098, 'train_steps_per_second': 1.944, 'total_flos': 1067878802522112.0, 'train_loss': 2.113566182454427, 'epoch': 3.0})


In [ ]:
import os

OUTPUT_DIR = "/content/nllb-lora-fr-lin-final"

# Sauvegarder les adaptateurs LoRA
model.save_pretrained(OUTPUT_DIR)

# Sauvegarder le tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

print("Modèle LoRA sauvegardé avec succès !")
print("Emplacement :", OUTPUT_DIR)

print("Fichiers sauvegardés :")
print(os.listdir(OUTPUT_DIR))

Modèle LoRA sauvegardé avec succès !
Emplacement : /content/nllb-lora-fr-lin-final
Fichiers sauvegardés :
['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'sentencepiece.bpe.model']


In [ ]:
import torch

def traduire_fr_lingala(texte):
    model.eval()

    inputs = tokenizer(
        texte,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("lin_Latn"),
            max_length=256,
            num_beams=4
        )

    traduction = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    return traduction


phrase = "Bonjour, comment allez-vous aujourd'hui ?"

resultat = traduire_fr_lingala(phrase)

print("Français :", phrase)
print("Lingala :", resultat)

Français : Bonjour, comment allez-vous aujourd'hui ?
Lingala : Mbote, ozali kosala nini lelo?


In [ ]:
import sacrebleu
from tqdm.auto import tqdm

model.eval()

predictions = []
references = []

for example in tqdm(splits["test"]):
    prediction = traduire_fr_lingala(example["source"])

    predictions.append(prediction)
    references.append(example["target"])

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [references]
)

print("===== ÉVALUATION DU MODÈLE LoRA =====")
print(f"BLEU : {bleu.score:.2f}")
print(f"chrF : {chrf.score:.2f}")

  0%|          | 0/500 [00:00<?, ?it/s]

===== ÉVALUATION DU MODÈLE LoRA =====
BLEU : 21.32
chrF : 51.13


In [ ]:
from transformers import AutoModelForSeq2SeqLM
import torch
import sacrebleu
from tqdm.auto import tqdm

print("Chargement du modèle NLLB original...")

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to("cuda" if torch.cuda.is_available() else "cpu")

base_model.eval()

target_language_id = tokenizer.convert_tokens_to_ids("lin_Latn")

def traduire_avec_modele(modele, texte):
    inputs = tokenizer(
        texte,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(modele.device)

    with torch.no_grad():
        outputs = modele.generate(
            **inputs,
            forced_bos_token_id=target_language_id,
            max_length=256,
            num_beams=4
        )

    return tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]


base_predictions = []
references = []

print("Évaluation du modèle original...")

for example in tqdm(splits["test"]):
    prediction = traduire_avec_modele(
        base_model,
        example["source"]
    )

    base_predictions.append(prediction)
    references.append(example["target"])

base_bleu = sacrebleu.corpus_bleu(
    base_predictions,
    [references]
)

base_chrf = sacrebleu.corpus_chrf(
    base_predictions,
    [references]
)

print("\n===== MODÈLE NLLB ORIGINAL =====")
print(f"BLEU : {base_bleu.score:.2f}")
print(f"chrF : {base_chrf.score:.2f}")

print("\n===== COMPARAISON =====")
print(f"LoRA   : BLEU 21.71 | chrF 51.44")
print(f"Original : BLEU {base_bleu.score:.2f} | chrF {base_chrf.score:.2f}")

Chargement du modèle NLLB original...
Évaluation du modèle original...


  0%|          | 0/500 [00:00<?, ?it/s]


===== MODÈLE NLLB ORIGINAL =====
BLEU : 16.24
chrF : 46.46

===== COMPARAISON =====
LoRA   : BLEU 21.71 | chrF 51.44
Original : BLEU 16.24 | chrF 46.46


In [ ]:
# Comparaison de quelques traductions

for i in range(10):
    phrase = splits["test"][i]["source"]
    reference = splits["test"][i]["target"]

    traduction_lora = traduire_fr_lingala(phrase)

    traduction_originale = traduire_avec_modele(
        base_model,
        phrase
    )

    print("=" * 70)
    print("FRANÇAIS :", phrase)
    print("RÉFÉRENCE :", reference)
    print("NLLB ORIGINAL :", traduction_originale)
    print("NLLB + LoRA :", traduction_lora)

FRANÇAIS : Elle me disait que certains jours, elle en voulait à la terre entière.
RÉFÉRENCE : Azalaki koloba na ngai ete mikolo misusu, aboyaki mokili mobimba.
NLLB ORIGINAL : Ayebisaki ngai ete mikolo mosusu azalaki kosilikela bato ya mokili mobimba.
NLLB + LoRA : Azalaki koyebisa ngai ete mikolo misusu, azalaki kolinga mokili mobimba.
FRANÇAIS : Elle est allée en France pour étudier l'art.
RÉFÉRENCE : Akendeki koyekola art o mboka France.
NLLB ORIGINAL : Akendaki na France mpo na koyekola mayemi.
NLLB + LoRA : Akendaki na France mpo na koyekola mosala ya ntɔki.
FRANÇAIS : Stop, attendez !
RÉFÉRENCE : Telema, zela !
NLLB ORIGINAL : Botika, bozela!
NLLB + LoRA : Botika, bozela!
FRANÇAIS : Le regard vide des yeux de poupées m'a toujours effrayé.
RÉFÉRENCE : Etaleli ya miso ya bana popi ebangisaka ngai.
NLLB ORIGINAL : Nazalaki ntango nyonso kobanga komona miso ya bana-popi mpamba.
NLLB + LoRA : Likanisi ya pamba ya miso ya bapota ezalaki kobangisa ngai tango nyonso.
FRANÇAIS : L'absence

In [ ]:
import sacrebleu

# Calculer un score BLEU individuel pour chaque phrase
erreurs = []

for i, example in enumerate(splits["test"]):
    reference = example["target"]
    prediction = traduire_fr_lingala(example["source"])

    score = sacrebleu.sentence_bleu(
        prediction,
        [reference]
    ).score

    erreurs.append({
        "source": example["source"],
        "reference": reference,
        "prediction": prediction,
        "bleu": score
    })

# Trier du score le plus faible au plus élevé
erreurs_triees = sorted(
    erreurs,
    key=lambda x: x["bleu"]
)

# Afficher les 30 cas les plus difficiles
for i, item in enumerate(erreurs_triees[:30], 1):
    print("=" * 70)
    print("CAS :", i)
    print("FRANÇAIS :", item["source"])
    print("RÉFÉRENCE :", item["reference"])
    print("LoRA :", item["prediction"])
    print("BLEU :", round(item["bleu"], 2))

CAS : 1
FRANÇAIS : Le FBI le surveille depuis 1987.
RÉFÉRENCE : FBI elandelaka ye uta mobu nkoto yoko na nkama libwa na ntuk mwambe na sambo (1987).
LoRA : FBI ezali kotala ye banda 1987.
BLEU : 1.67
CAS : 2
FRANÇAIS : La force athlétique est un sport où les athlètes en compétition doivent soulever un poids maximum cumulé de trois épreuves de poids, la flexion des jambes, le développé-couché et le soulevé de terre.
RÉFÉRENCE : Makasi ya atletike eza masano epayi wapi bato ba tombolaka kilo esangisami elongo ya mimekano misatu ya kilo, oyo ya makolo ya ko ngunzama, oyo  ya kosala na kolala na se mpe oyo ya kotombola kauta na mabélé.
LoRA : Nguya ya lisano ya masano ezali lisano oyo basani ya masano basengeli kotombola bozito ya monene ya masano misato ya masano ya masano ya masano ya masano, kobenda makolo, kolala mpe kotombola mabele.
BLEU : 1.78
CAS : 3
FRANÇAIS : Il est tellement ouvert aux impressions qui effleurent à peine le commun des esprits doués et vibrants qu’il trouve cette 

In [ ]:
from collections import Counter

print("===== QUALITÉ DU DATASET =====")

print("Nombre total :", len(ds))

# Détecter les doublons français
sources = ds["source"]
doublons = [
    phrase for phrase, count in Counter(sources).items()
    if count > 1
]

print("Phrases françaises en doublon :", len(doublons))

# Vérifier les paires vides
vides = [
    i for i, example in enumerate(ds)
    if not example["source"].strip()
    or not example["target"].strip()
]

print("Paires vides :", len(vides))

# Statistiques de longueur
longueurs_fr = [
    len(example["source"].split()) for example in ds
]

longueurs_lin = [
    len(example["target"].split()) for example in ds
]

print("Moyenne mots français :", round(sum(longueurs_fr)/len(longueurs_fr), 2))
print("Moyenne mots lingala :", round(sum(longueurs_lin)/len(longueurs_lin), 2))

# Afficher quelques doublons
print("\nExemples de doublons :")
for phrase in doublons[:10]:
    print("-", phrase)

===== QUALITÉ DU DATASET =====
Nombre total : 5000
Phrases françaises en doublon : 5
Paires vides : 0
Moyenne mots français : 13.61
Moyenne mots lingala : 13.02

Exemples de doublons :
- Je me casse.
- Quand le chat n'est pas là, les souris dansent.
- De mon côté, je ne voyage pas pour aller quelque part. Je voyage pour voyager. Le principal est de se bouger.
- C'est seulement quand je ne peux pas dormir la nuit que le tic-tac de l'horloge devient assez bruyant pour m'embêter.
- Il est doux et noble de mourir pour la patrie.


In [ ]:
from collections import defaultdict

groupes = defaultdict(list)

for exemple in ds:
    groupes[exemple["source"]].append(exemple["target"])

print("===== ANALYSE DES DOUBLONS =====")

for phrase, traductions in groupes.items():
    if len(traductions) > 1:
        print("\nFRANÇAIS :", phrase)
        for i, traduction in enumerate(traductions, start=1):
            print(f"Traduction {i} :", traduction)

===== ANALYSE DES DOUBLONS =====

FRANÇAIS : Je me casse.
Traduction 1 : Nakeyi.
Traduction 2 : Nakeyi.

FRANÇAIS : Quand le chat n'est pas là, les souris dansent.
Traduction 1 : Ntango nyawu azali te, ba mpuku babinaka.
Traduction 2 : Ntangu nyawu azali te, ba mpuku babinaka.

FRANÇAIS : De mon côté, je ne voyage pas pour aller quelque part. Je voyage pour voyager. Le principal est de se bouger.
Traduction 1 : Na ngambo nangayi, nazali kosala mobembo te mpona kokende esika moko. Nazo sala mobembo kaka pona sala. Ya motuya ezali koningana.
Traduction 2 : Na oyo etali nagayi, nazali kosala mobembo te pona elengi ya kokende bisika songolo. Nazali kosala mobembo kaka tika nasala. Motuya ezali kotambola.

FRANÇAIS : C'est seulement quand je ne peux pas dormir la nuit que le tic-tac de l'horloge devient assez bruyant pour m'embêter.
Traduction 1 : Kaka na ngonga nazangi mpongi na butu nde tiki-taki (tic tac) ya montele monene ekomaka makelele mpo na kotungisa ngai.
Traduction 2 : Kaka na nt

In [ ]:
# 1. Convertir le Dataset Hugging Face en Pandas
df = ds.to_pandas()

# 2. Supprimer les paires strictement identiques
df_nettoye = df.drop_duplicates(
    subset=["source", "target"]
)

# 3. Reconvertir en Dataset Hugging Face
from datasets import Dataset

ds_nettoye = Dataset.from_pandas(
    df_nettoye,
    preserve_index=False
)

# 4. Afficher les résultats
print("===== NETTOYAGE DU DATASET =====")
print("Dataset original :", len(ds))
print("Dataset nettoyé :", len(ds_nettoye))
print("Paires supprimées :", len(ds) - len(ds_nettoye))

===== NETTOYAGE DU DATASET =====
Dataset original : 5000
Dataset nettoyé : 4999
Paires supprimées : 1


In [ ]:
import random
from datasets import DatasetDict

random.seed(42)

indices = list(range(len(ds_nettoye)))
random.shuffle(indices)

n = len(indices)
n_train = int(n * 0.80)
n_val = int(n * 0.10)

splits_nettoyes = DatasetDict({
    "train": ds_nettoye.select(indices[:n_train]),
    "validation": ds_nettoye.select(
        indices[n_train:n_train + n_val]
    ),
    "test": ds_nettoye.select(
        indices[n_train + n_val:]
    )
})

print("===== NOUVEAU DÉCOUPAGE =====")
print("Train :", len(splits_nettoyes["train"]))
print("Validation :", len(splits_nettoyes["validation"]))
print("Test :", len(splits_nettoyes["test"]))

===== NOUVEAU DÉCOUPAGE =====
Train : 3999
Validation : 499
Test : 501


In [ ]:
def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["source"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_nettoye = splits_nettoyes.map(
    tokenize_batch,
    batched=True,
    remove_columns=splits_nettoyes["train"].column_names
)

print("===== TOKENISATION TERMINÉE =====")
print("Train :", len(tokenized_nettoye["train"]))
print("Validation :", len(tokenized_nettoye["validation"]))
print("Test :", len(tokenized_nettoye["test"]))
print("Colonnes :", tokenized_nettoye["train"].column_names)

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

# Charger le modèle NLLB original
model_nettoye = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

# Configurer la langue cible : Lingala
target_language_id = tokenizer.convert_tokens_to_ids("lin_Latn")

model_nettoye.config.forced_bos_token_id = target_language_id
model_nettoye.generation_config.forced_bos_token_id = target_language_id

# Configuration LoRA
lora_config_nettoye = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Appliquer LoRA
model_nettoye = get_peft_model(
    model_nettoye,
    lora_config_nettoye
)

# Afficher les paramètres entraînables
model_nettoye.print_trainable_parameters()

trainable params: 1,179,648 || all params: 616,253,440 || trainable%: 0.1914


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args_nettoye = Seq2SeqTrainingArguments(
    output_dir="./nllb-lora-fr-lin-nettoye",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator_nettoye = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_nettoye,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

print("Data Collator créé avec succès !")

Data Collator créé avec succès !


In [ ]:
print("Variables disponibles :")
for nom in ["splits", "train_dataset", "eval_dataset",
            "dataset_nettoye", "tokenized_nettoye",
            "tokenized", "tokenizer"]:
    print(nom, ":", nom in globals())

Variables disponibles :
splits : True
train_dataset : False
eval_dataset : False
dataset_nettoye : False
tokenized_nettoye : False
tokenized : True
tokenizer : True


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model_nettoye,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator_nettoye,
    tokenizer=tokenizer
)

print("Trainer créé avec succès !")

/usr/local/lib/python3.13/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Trainer créé avec succès !


In [ ]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss
1,2.089700,2.063485
2,2.030000,2.039079
3,2.035100,2.031129


In [ ]:
import torch
import sacrebleu
from tqdm.auto import tqdm

model_nettoye.eval()

def traduire_modele_nettoye(texte):
    inputs = tokenizer(
        texte,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model_nettoye.device)

    with torch.no_grad():
        outputs = model_nettoye.generate(
            **inputs,
            forced_bos_token_id=target_language_id,
            max_length=256,
            num_beams=4
        )

    return tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]


predictions_nettoyees = []
references_nettoyees = []

for exemple in tqdm(splits_nettoyes["test"]):
    prediction = traduire_modele_nettoye(exemple["source"])

    predictions_nettoyees.append(prediction)
    references_nettoyees.append(exemple["target"])


bleu_nettoye = sacrebleu.corpus_bleu(
    predictions_nettoyees,
    [references_nettoyees]
)

chrf_nettoye = sacrebleu.corpus_chrf(
    predictions_nettoyees,
    [references_nettoyees]
)

print("===== ÉVALUATION DU NOUVEAU MODÈLE =====")
print("BLEU :", round(bleu_nettoye.score, 2))
print("chrF :", round(chrf_nettoye.score, 2))

  0%|          | 0/501 [00:00<?, ?it/s]

===== ÉVALUATION DU NOUVEAU MODÈLE =====
BLEU : 19.9
chrF : 50.81


In [ ]:
import torch
import sacrebleu
from tqdm.auto import tqdm

# Vérifier que les deux modèles sont disponibles
model.eval()
model_nettoye.eval()

# Utiliser exactement le même jeu de test
references_communes = splits_nettoyes["test"]["target"]
sources_communes = splits_nettoyes["test"]["source"]

predictions_premier = []
predictions_second = []

for texte in tqdm(sources_communes):

    # Première expérience LoRA
    prediction_1 = traduire_avec_modele(model, texte)

    # Deuxième expérience LoRA
    prediction_2 = traduire_modele_nettoye(texte)

    predictions_premier.append(prediction_1)
    predictions_second.append(prediction_2)


# Calcul des scores BLEU
bleu_1 = sacrebleu.corpus_bleu(
    predictions_premier,
    [references_communes]
)

bleu_2 = sacrebleu.corpus_bleu(
    predictions_second,
    [references_communes]
)

# Calcul des scores chrF
chrf_1 = sacrebleu.corpus_chrf(
    predictions_premier,
    [references_communes]
)

chrf_2 = sacrebleu.corpus_chrf(
    predictions_second,
    [references_communes]
)


print("===== COMPARAISON SUR LE MÊME TEST =====")

print("\nPREMIER MODÈLE")
print("BLEU :", round(bleu_1.score, 2))
print("chrF :", round(chrf_1.score, 2))

print("\nDEUXIÈME MODÈLE")
print("BLEU :", round(bleu_2.score, 2))
print("chrF :", round(chrf_2.score, 2))

  0%|          | 0/501 [00:00<?, ?it/s]

===== COMPARAISON SUR LE MÊME TEST =====

PREMIER MODÈLE
BLEU : 20.16
chrF : 50.81

DEUXIÈME MODÈLE
BLEU : 19.9
chrF : 50.81


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/CVBoost/checkpoints/nllb-lora-lin-fra"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Dossier de sauvegarde :", OUTPUT_DIR)
print("Dossier accessible :", os.path.exists(OUTPUT_DIR))

Dossier de sauvegarde : /content/drive/MyDrive/CVBoost/checkpoints/nllb-lora-lin-fra
Dossier accessible : True


In [ ]:
!pip uninstall -y peft
!pip install peft==0.17.1

Found existing installation: peft 0.12.0
Uninstalling peft-0.12.0:
  Successfully uninstalled peft-0.12.0


In [ ]:
import peft
import torch

print("PEFT :", peft.__version__)
print("GPU disponible :", torch.cuda.is_available())

PEFT : 0.17.1
GPU disponible : True


In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/CVBoost/checkpoints/nllb-lora-lin-fra"

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Modèle sauvegardé dans Google Drive !")
print(os.listdir(OUTPUT_DIR))

Modèle sauvegardé dans Google Drive !
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'sentencepiece.bpe.model', 'tokenizer.json', 'training_args.bin']


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

MODEL_NAME = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR = "/content/drive/MyDrive/CVBoost/checkpoints/nllb-lora-lin-fra"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Charger le tokenizer
tokenizer_test = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="fra_Latn"
)

# Charger le modèle de base
base_model_test = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

# Recharger les poids LoRA entraînés
model_test = PeftModel.from_pretrained(
    base_model_test,
    OUTPUT_DIR
)

model_test.eval()

target_id = tokenizer_test.convert_tokens_to_ids("lin_Latn")

print("Modèle LoRA rechargé sur :", device)

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Modèle LoRA rechargé sur : cuda


In [ ]:
def traduire_lingala(texte):
    inputs = tokenizer_test(
        texte,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model_test.generate(
            **inputs,
            forced_bos_token_id=target_id,
            max_length=256,
            num_beams=4
        )

    return tokenizer_test.decode(
        outputs[0],
        skip_special_tokens=True
    )


phrases = [
    "Bonjour, comment allez-vous aujourd'hui ?",
    "Je suis développeuse informatique.",
    "Bienvenue sur notre plateforme.",
    "Nous vous aidons à trouver un emploi.",
    "Merci beaucoup pour votre confiance."
]

for phrase in phrases:
    print("FR :", phrase)
    print("LIN :", traduire_lingala(phrase))
    print("-" * 60)

FR : Bonjour, comment allez-vous aujourd'hui ?
LIN : Mbote, ozali kosala nini lelo?
------------------------------------------------------------
FR : Je suis développeuse informatique.
LIN : Nazali mobongisi ya ba ordinatɛrɛ.
------------------------------------------------------------
FR : Bienvenue sur notre plateforme.
LIN : Boyei malamu na site na biso.
------------------------------------------------------------
FR : Nous vous aidons à trouver un emploi.
LIN : Tosalisaka yo ozwa mosala.
------------------------------------------------------------
FR : Merci beaucoup pour votre confiance.
LIN : Natondi yo mingi mpo na botyeli yo motema.
------------------------------------------------------------


In [ ]:
def traduire_avec_modele(modele, texte):
    inputs = tokenizer_test(
        texte,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = modele.generate(
            **inputs,
            forced_bos_token_id=target_id,
            max_length=256,
            num_beams=4
        )

    return tokenizer_test.decode(
        outputs[0],
        skip_special_tokens=True
    )


phrases = [
    "Bonjour, comment allez-vous aujourd'hui ?",
    "Je suis développeuse informatique.",
    "Bienvenue sur notre plateforme.",
    "Nous vous aidons à trouver un emploi.",
    "Merci beaucoup pour votre confiance."
]

for phrase in phrases:
    print("\nFR :", phrase)
    print("BASE :", traduire_avec_modele(base_model_test, phrase))
    print("LORA :", traduire_avec_modele(model_test, phrase))
    print("-" * 60)


FR : Bonjour, comment allez-vous aujourd'hui ?
BASE : Mbote, ozali kosala nini lelo?
LORA : Mbote, ozali kosala nini lelo?
------------------------------------------------------------

FR : Je suis développeuse informatique.
BASE : Nazali mobongisi ya ba ordinatɛrɛ.
LORA : Nazali mobongisi ya ba ordinatɛrɛ.
------------------------------------------------------------

FR : Bienvenue sur notre plateforme.
BASE : Boyei malamu na site na biso.
LORA : Boyei malamu na site na biso.
------------------------------------------------------------

FR : Nous vous aidons à trouver un emploi.
BASE : Tosalisaka yo ozwa mosala.
LORA : Tosalisaka yo ozwa mosala.
------------------------------------------------------------

FR : Merci beaucoup pour votre confiance.
BASE : Natondi yo mingi mpo na botyeli yo motema.
LORA : Natondi yo mingi mpo na botyeli yo motema.
------------------------------------------------------------


In [ ]:
# Charger un modèle de base propre, sans LoRA
base_clean = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

base_clean.eval()

def comparer_modeles(texte):
    inputs = tokenizer_test(
        texte,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        resultat_base = base_clean.generate(
            **inputs,
            forced_bos_token_id=target_id,
            max_length=256,
            num_beams=4
        )

        resultat_lora = model_test.generate(
            **inputs,
            forced_bos_token_id=target_id,
            max_length=256,
            num_beams=4
        )

    return (
        tokenizer_test.decode(resultat_base[0], skip_special_tokens=True),
        tokenizer_test.decode(resultat_lora[0], skip_special_tokens=True)
    )

for phrase in phrases:
    base, lora = comparer_modeles(phrase)
    print("\nFR :", phrase)
    print("BASE :", base)
    print("LORA :", lora)
    print("-" * 60)


FR : Bonjour, comment allez-vous aujourd'hui ?
BASE : Bonjour, ozali kosala nini lelo?
LORA : Mbote, ozali kosala nini lelo?
------------------------------------------------------------

FR : Je suis développeuse informatique.
BASE : Nazali mobongisi ya ordinatɛrɛ.
LORA : Nazali mobongisi ya ba ordinatɛrɛ.
------------------------------------------------------------

FR : Bienvenue sur notre plateforme.
BASE : Tóya na site na biso.
LORA : Boyei malamu na site na biso.
------------------------------------------------------------

FR : Nous vous aidons à trouver un emploi.
BASE : Tosalisaka yo ozwa mosala.
LORA : Tosalisaka yo ozwa mosala.
------------------------------------------------------------

FR : Merci beaucoup pour votre confiance.
BASE : Matɔndi mingi mpo na botyeli ngai motema.
LORA : Natondi yo mingi mpo na botyeli yo motema.
------------------------------------------------------------


In [ ]:
!pip install -q sacrebleu

In [ ]:
import torch
import sacrebleu
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

# Préparer les données de test
test_dataset = tokenized["test"]

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer_test,
    model=base_clean,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if device == "cuda" else None
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collator
)

def evaluer_modele(modele, nom):
    modele.eval()
    predictions = []
    references = []

    for batch in test_loader:
        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key in ["input_ids", "attention_mask"]
        }

        with torch.no_grad():
            generated = modele.generate(
                **inputs,
                forced_bos_token_id=target_id,
                max_length=256,
                num_beams=4
            )

        predictions.extend(
            tokenizer_test.batch_decode(
                generated,
                skip_special_tokens=True
            )
        )

        labels = labels.cpu().numpy()
        labels[labels == -100] = tokenizer_test.pad_token_id

        references.extend(
            tokenizer_test.batch_decode(
                labels,
                skip_special_tokens=True
            )
        )

    bleu = sacrebleu.corpus_bleu(
        predictions, [references]
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions, [references]
    ).score

    print(f"\nRésultats : {nom}")
    print(f"BLEU : {bleu:.2f}")
    print(f"chrF : {chrf:.2f}")

    return {"BLEU": bleu, "chrF": chrf}


# Évaluer le modèle de base
resultats_base = evaluer_modele(base_clean, "NLLB de base")

# Évaluer le modèle LoRA
resultats_lora = evaluer_modele(model_test, "NLLB + LoRA")

# Comparaison finale
print("\n========== COMPARAISON ==========")
print("BASE :", resultats_base)
print("LoRA :", resultats_lora)

print("\nDifférence BLEU :",
      round(resultats_lora["BLEU"] - resultats_base["BLEU"], 2))

print("Différence chrF :",
      round(resultats_lora["chrF"] - resultats_base["chrF"], 2))


Résultats : NLLB de base
BLEU : 16.47
chrF : 46.49

Résultats : NLLB + LoRA
BLEU : 21.78
chrF : 51.37

========== COMPARAISON ==========
BASE : {'BLEU': 16.469584924429643, 'chrF': 46.4944135612476}
LoRA : {'BLEU': 21.784399543643662, 'chrF': 51.37008518905386}

Différence BLEU : 5.31
Différence chrF : 4.88
